## Step 1 Install

In [1]:
!pip install pypdf sentence-transformers faiss-cpu transformers

   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ------- -------------------------------- 2.9/16.1 MB 13.9 MB/s eta 0:00:01
   ------------- -------------------------- 5.5/16.1 MB 13.4 MB/s eta 0:00:01
   --------------------- ------------------ 8.7/16.1 MB 13.8 MB/s eta 0:00:01
   ----------------------------- ---------- 11.8/16.1 MB 14.2 MB/s eta 0:00:01
   ------------------------------------ --- 14.7/16.1 MB 14.0 MB/s eta 0:00:01
   ---------------------------------------  16.0/16.1 MB 14.2 MB/s eta 0:00:01
   ---------------------------------------- 16.1/16.1 MB 11.0 MB/s  0:00:01

   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   ---------------------------------------- 0/2 [pypdf]
   -------------------- ------------------- 1/2 [faiss-cpu]
   -------------------- ------------------- 1/2 [faiss-cpu]
   -------------------- ------------------- 1/2 [faiss-cpu]
   -------------------- ----------

## Step 2 Imports

In [2]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

C:\Users\SivaGuduri\Desktop\learn-ML\dl-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3 Load embedding model
This converts text chunks → vectors.

In [3]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4932.11it/s]


## Step 4 Load generation model

In [4]:
generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 76/76 [00:00<00:00, 6928.82it/s]


## Step 5 Create sample PDF text manually first (before actual PDF)

In [5]:
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Natural language processing helps computers understand text.",
    "Computer vision helps machines understand images."
]

## Step 6 Create embeddings

In [6]:
doc_embeddings = embed_model.encode(documents)

## Step 7 Create FAISS vector DB
Store document vectors for fast similarity search

In [7]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings))

## Step 8 Ask a question

In [10]:
query = "What is NLP?"
query_embedding = embed_model.encode([query])

## Step 9 Search in FAISS

In [11]:
distances, indices = index.search(
    np.array(query_embedding),
    k=2
)

print(indices)

[[2 0]]


## Step 10 Retrieve matching docs

In [12]:
retrieved_docs = [documents[i] for i in indices[0]]

print(retrieved_docs)

['Natural language processing helps computers understand text.', 'Machine learning is a subset of artificial intelligence.']


## Step 11 Generate answer using retrieved context

In [13]:
context = " ".join(retrieved_docs)

prompt = f"""
Context: {context}

Question: {query}

Answer:
"""

In [14]:
result = generator(
    prompt,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=1.2
)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_s


Context: Natural language processing helps computers understand text. Machine learning is a subset of artificial intelligence.

Question: What is NLP?

Answer:
NLP has three main features and can be applied in many applications to the computer operating system including machine-learning, speech recognition, search engine optimization (SEO) and computational algorithms such as Google Translate or Deep Learning.[1] The two are related concepts but have different meanings; they differ widely among people who do not use real languages like English,[2][3]. A natural language should also


## Read Real PDF

In [15]:
!pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 13.6 MB/s  0:00:00


In [16]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

doc = SimpleDocTemplate("sample_ai_notes.pdf")
styles = getSampleStyleSheet()

content = [
    Paragraph("Machine Learning is a subset of Artificial Intelligence.", styles["Normal"]),
    Paragraph("Deep Learning uses neural networks with many hidden layers.", styles["Normal"]),
    Paragraph("Natural Language Processing helps computers understand human language.", styles["Normal"]),
    Paragraph("Computer Vision helps machines understand images and videos.", styles["Normal"]),
    Paragraph("Transformers are modern architectures used in GPT and BERT.", styles["Normal"]),
]

doc.build(content)

print("PDF created!")

PDF created!


## 1 Read the PDF

In [18]:
reader = PdfReader("sample_ai_notes.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text)

Machine Learning is a subset of Artificial Intelligence.
Deep Learning uses neural networks with many hidden layers.
Natural Language Processing helps computers understand human language.
Computer Vision helps machines understand images and videos.
Transformers are modern architectures used in GPT and BERT.



## 2 Split into chunks

In [19]:
chunk_size = 200

chunks = [
    text[i:i+chunk_size]
    for i in range(0, len(text), chunk_size)
]

print(chunks)

['Machine Learning is a subset of Artificial Intelligence.\nDeep Learning uses neural networks with many hidden layers.\nNatural Language Processing helps computers understand human language.\nComputer Vis', 'ion helps machines understand images and videos.\nTransformers are modern architectures used in GPT and BERT.\n']


## 3 Create embeddings

In [20]:
chunk_embeddings = embed_model.encode(chunks)

## 4 Store in FAISS

In [21]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(chunk_embeddings))

## 5 Ask a question

In [22]:
query = "What are transformers?"

In [23]:
query_embedding = embed_model.encode([query])

## 6 Search relevant chunks

In [24]:
distances, indices = index.search(
    np.array(query_embedding),
    k=2
)

retrieved_chunks = [chunks[i] for i in indices[0]]

print(retrieved_chunks)

['ion helps machines understand images and videos.\nTransformers are modern architectures used in GPT and BERT.\n', 'Machine Learning is a subset of Artificial Intelligence.\nDeep Learning uses neural networks with many hidden layers.\nNatural Language Processing helps computers understand human language.\nComputer Vis']


## 7 Generate answer

In [25]:
context = " ".join(retrieved_chunks)

prompt = f"""
Use this context to answer:

{context}

Question: {query}

Answer:
"""

In [26]:
result = generator(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=1.2
)

print(result[0]["generated_text"])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use this context to answer:

ion helps machines understand images and videos.
Transformers are modern architectures used in GPT and BERT.
 Machine Learning is a subset of Artificial Intelligence.
Deep Learning uses neural networks with many hidden layers.
Natural Language Processing helps computers understand human language.
Computer Vis

Question: What are transformers?

Answer:
The term "supervised" comes from the word super-trained, which refers specifically by artificial intelligence for machine learning tasks (e., processing data) that can be trained on real or simulated conditions using sophisticated algorithms such as deep training procedures designed at low cost compared against current systems based only on finite datasets where memory access exceeds capacity because computations aren't fast enough; then there are lots more computing power required once you learn how to read files online without having to rely heavily upon large databases like MongoDB
